In [ ]:
"""
Study area map — La Araucanía, Chile
Two-panel figure:
  Left  : Chile overview — regional borders, Wallmapu (clipped to both coasts,
           blurred N/S edges), legend, leader lines to detail panel

Requirements:
    pip install cartopy matplotlib geopandas shapely requests pyproj

Run:    python araucania_map.py
Output: Araucania_Study_Area_Map.png / .pdf
"""

import warnings, zipfile, io, json, os
from pathlib import Path
warnings.filterwarnings("ignore")

import requests
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Polygon, box
from shapely.ops import unary_union

# ── cache ──────────────────────────────────────────────────────────────────
CACHE = Path("./map_data")
CACHE.mkdir(exist_ok=True)

# ── typography ─────────────────────────────────────────────────────────────
FONT   = "DejaVu Sans"
FS_AX  = 7.5    # axis ticks, captions
FS_LK  = 8.5    # lake labels, legend
FS_CT  = 10.5   # towns, context labels
FS_TP  = 12.0   # Temuco capital, overlay labels
FS_TTL = 11.0   # panel titles

# ── palette ────────────────────────────────────────────────────────────────
LAND       = "#eae6d8"
OCEAN      = "#a8cce0"
CHILE_FILL = "#ddd8c4"
ARG_FILL   = "#d8d2bf"
ARA_FILL   = "#2e7ea8"
ARA_ALPHA  = 0.28
ARA_EDGE   = "#164f6e"
WALL_FILL  = "#2e7d32"
WALL_ALPHA = 0.38
WALL_EDGE  = "#1b5e20"
REG_EDGE   = "#a09080"
LAKE_COL   = "#82b8d8"
LAKE_EDGE  = "#2a6898"
VOL_COL    = "#d32f2f"
VOL_LBL    = "#b71c1c"
CITY_CAP   = "#000000"
CITY_TOWN  = "#000000"
CITY_LBL   = "#000000"
WS         = "white"

# ── GVP-verified volcano coordinates ──────────────────────────────────────
VOLCANOES = [
    ("Llaima",        -71.730, -38.692, 3125),
    ("Sierra Nevada", -71.580, -38.580, 2554),
    ("Tolhuaca",      -71.670, -38.290, 2806),
    ("Lonquimay",     -71.586, -38.380, 2865),
    ("Sollipulli",    -71.520, -38.970, 2282),
    ("Villarrica",    -71.940, -39.420, 2847),
]

VOL_OFF = {
    "Llaima":        ( 0.05, -0.05, "left"),
    "Sierra Nevada": ( 0.05,  0.04, "left"),
    "Tolhuaca":      ( 0.05,  0.04, "left"),
    "Lonquimay":     ( 0.05, -0.05, "left"),
    "Sollipulli":    ( 0.05,  0.04, "left"),
    "Villarrica":    (-0.05, -0.05, "right"),
}

# Cities: name, lon, lat, capital, dx, dy, ha
CITIES = [
    ("Temuco",        -72.590, -38.735, True,  -0.04, -0.04, "right"),
    ("Vilcún",        -72.224, -38.647, False,  0.03,  0.03, "left"),
    ("Melipeuco",     -71.876, -38.852, False, -0.04, -0.03, "right"),
    ("Malalcahuello", -71.573, -38.453, False, -0.03,  0.04, "right"),   # closer, north
    ("Pucón",         -71.978, -39.272, False,  0.04,  0.03, "left"),
    ("Villarrica",    -72.229, -39.283, False, -0.04, -0.03, "right"),
    ("Cunco",         -72.028, -38.930, False,  0.00,  0.10, "center"),
    ("Curacautín",    -71.883, -38.433, False, -0.03, -0.04, "right"),   # closer, SW
    ("Icalma",        -71.285, -38.811, False,  0.04,  0.03, "left"),
]

LAKE_LABELS = [
    (-72.11, -39.30, "L. Villarrica"),
]

# Araucanía window
ARA = dict(w=-73.7, e=-70.7, s=-39.75, n=-37.45)


# ── Natural Earth downloader ───────────────────────────────────────────────
def download_ne(scale, category, name):
    fname = f"ne_{scale}_{name}"
    zp    = CACHE / f"{fname}.zip"
    exdir = CACHE / fname
    done  = exdir / ".done"
    if not done.exists():
        for url in [
            f"https://naciscdn.org/naturalearth/{scale}/{category}/{fname}.zip",
            f"https://naturalearth.s3.amazonaws.com/{scale}_{category}/{fname}.zip",
            f"https://github.com/nvkelso/natural-earth-vector/raw/master/zips/{fname}.zip",
        ]:
            try:
                print(f"  GET {url}")
                r = requests.get(url, timeout=120)
                if r.status_code == 200 and len(r.content) > 5000:
                    zp.write_bytes(r.content)
                    exdir.mkdir(exist_ok=True)
                    with zipfile.ZipFile(zp) as z:
                        z.extractall(exdir)
                    done.touch()
                    print(f"  ✓ {fname}")
                    break
            except Exception as e:
                print(f"  fail: {e}")
        if not done.exists():
            raise RuntimeError(f"Could not download {fname}")
    return gpd.read_file(next(exdir.glob("**/*.shp")))


_PROJ = ccrs.PlateCarree()

def stroke(lw=2.5):
    return [pe.withStroke(linewidth=lw, foreground=WS)]


def draw_poly(ax, geom, z=3, **kw):
    fc = kw.pop("facecolor", kw.pop("fc", "none"))
    ec = kw.pop("edgecolor", kw.pop("ec", "none"))
    lw = kw.pop("linewidth", kw.pop("lw", 0.5))
    ls = kw.pop("linestyle", kw.pop("ls", "-"))
    al = kw.pop("alpha", 1.0)
    parts = list(geom.geoms) if geom.geom_type in (
        "MultiPolygon", "GeometryCollection") else [geom]
    for p in parts:
        if p.geom_type != "Polygon" or p.is_empty:
            continue
        xs, ys = p.exterior.xy
        ax.fill(list(xs), list(ys), fc=fc, ec=ec, lw=lw, ls=ls,
                alpha=al, transform=_PROJ, zorder=z)
        for ring in p.interiors:
            ix, iy = ring.xy
            ax.fill(list(ix), list(iy), fc=OCEAN,
                    ec="none", alpha=1.0, transform=_PROJ, zorder=z + 0.1)


def north_arrow(ax, x=0.95, y=0.07):
    ax.annotate("", xy=(x, y + 0.07), xytext=(x, y),
                xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="black",
                                lw=1.5, mutation_scale=12))
    ax.text(x, y + 0.085, "N", transform=ax.transAxes,
            ha="center", va="bottom",
            fontsize=FS_CT, fontweight="bold", fontfamily=FONT)


def scale_bar(ax, lon0, lat0, km):
    deg = km / (111.0 * abs(np.cos(np.radians(lat0))))
    for i, col in enumerate(["black", "white"]):
        ax.plot([lon0 + i * deg / 2, lon0 + (i+1) * deg / 2],
                [lat0, lat0], color=col, lw=5,
                solid_capstyle="butt", transform=_PROJ, zorder=18,
                path_effects=[pe.withStroke(linewidth=7, foreground="black")])
    for val, x in [(0, lon0), (km//2, lon0+deg/2), (km, lon0+deg)]:
        ax.text(x, lat0 - 0.14, str(val), ha="center", va="top",
                fontsize=FS_AX, fontfamily=FONT,
                transform=_PROJ, zorder=19, path_effects=stroke(1.5))
    ax.text(lon0 + deg*1.06, lat0 - 0.14, "km",
            ha="left", va="top", fontsize=FS_AX, fontfamily=FONT,
            transform=_PROJ, zorder=19, path_effects=stroke(1.5))


# ── Wallmapu with blurred N/S edges ───────────────────────────────────────
def draw_wallmapu_blurred(ax, land_geom, bbox, n_steps=12):
    lat_n, lat_s = bbox[3], bbox[1]
    fade_deg = 1.2

    core_bbox = box(bbox[0], lat_s + fade_deg, bbox[2], lat_n - fade_deg)
    core = core_bbox.intersection(land_geom)
    draw_poly(ax, core, z=7, fc=WALL_FILL, ec="none", alpha=WALL_ALPHA)

    for i in range(n_steps):
        frac = i / n_steps
        alpha_val = WALL_ALPHA * (1 - frac)
        strip_h = fade_deg / n_steps

        n_bot = lat_n - fade_deg + i * strip_h
        n_top = n_bot + strip_h
        n_strip = box(bbox[0], n_bot, bbox[2], n_top).intersection(land_geom)
        if not n_strip.is_empty:
            draw_poly(ax, n_strip, z=7, fc=WALL_FILL, ec="none", alpha=alpha_val)

        s_top = lat_s + fade_deg - i * strip_h
        s_bot = s_top - strip_h
        s_strip = box(bbox[0], s_bot, bbox[2], s_top).intersection(land_geom)
        if not s_strip.is_empty:
            draw_poly(ax, s_strip, z=7, fc=WALL_FILL, ec="none", alpha=alpha_val)


# ══════════════════════════════════════════════════════════════════════════
def make_figure():
    print("Downloading data …")
    countries = download_ne("10m", "cultural", "admin_0_countries")
    regions   = download_ne("10m", "cultural", "admin_1_states_provinces")
    lakes_ne  = download_ne("10m", "physical",  "lakes")

    chile     = countries[countries["NAME"] == "Chile"]
    arg       = countries[countries["NAME"] == "Argentina"]
    chile_reg = regions[regions["admin"] == "Chile"]
    ara_reg   = chile_reg[
        chile_reg["name"].str.contains("raucan", case=False, na=False)]

    land_union = unary_union(list(chile.geometry) + list(arg.geometry))

    WALL_BBOX = (-82.0, -43.5, -54.0, -33.45)
    wallmapu_full = box(*WALL_BBOX).intersection(land_union)

    ara_buf    = box(ARA["w"]-0.4, ARA["s"]-0.4, ARA["e"]+0.4, ARA["n"]+0.4)
    lakes_clip = lakes_ne[lakes_ne.geometry.intersects(ara_buf)]

    # ── Figure ─────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(17, 10))
    from matplotlib.gridspec import GridSpec
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.45],
                  left=0.03, right=0.97, bottom=0.07, top=0.92, wspace=0.03)
    ax1 = fig.add_subplot(gs[0], projection=_PROJ)
    ax2 = fig.add_subplot(gs[1], projection=_PROJ)

    # ══════════════════════════════════════════════════════════════════
    #  Panel A — Chile overview
    #  Extent shifted east: less Pacific, full Argentina visible
    # ══════════════════════════════════════════════════════════════════
    ax = ax1
    ax.set_extent([-77, -52, -56, -17], crs=_PROJ)   # was [-82,-60]

    ax.add_feature(cfeature.OCEAN.with_scale("50m"),
                   facecolor=OCEAN, zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("50m"),
                   facecolor=LAND, edgecolor="none", zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"),
                   linewidth=0.5, edgecolor="#777", zorder=5)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"),
                   linewidth=0.6, edgecolor="#aaa",
                   linestyle="--", zorder=5)

    # Chile country outline — bold, drawn on top of generic borders
    chile.plot(ax=ax, facecolor="none", edgecolor="#2a2a2a",
               linewidth=1.8, zorder=7, transform=_PROJ)

    # Chilean regional borders
    for _, row in chile_reg.iterrows():
        draw_poly(ax, row.geometry, z=6,
                  fc="none", ec=REG_EDGE, lw=0.4, ls="-")

    # Wallmapu
    draw_wallmapu_blurred(ax, wallmapu_full, WALL_BBOX)

    # Araucanía highlight
    for _, row in ara_reg.iterrows():
        draw_poly(ax, row.geometry, z=8,
                  fc=ARA_FILL, ec=ARA_EDGE, lw=1.4, alpha=0.55)

    # Red inset box
    ax.add_patch(mpatches.Rectangle(
        (ARA["w"], ARA["s"]),
        ARA["e"]-ARA["w"], ARA["n"]-ARA["s"],
        fc="none", ec="#d32f2f", lw=1.6,
        transform=_PROJ, zorder=9))

    # Araucanía label
    ax.text(-71.5, -37.85, "La Araucanía",
            fontsize=FS_TP, fontweight="bold", fontfamily=FONT,
            color="white", ha="center", va="center",
            transform=_PROJ, zorder=10,
            bbox=dict(boxstyle="round,pad=0.22", fc=ARA_EDGE,
                      alpha=0.88, ec="none"))

    # Wallmapu label — Argentina side, upper portion of shading
    ax.text(-67.5, -35.5, "Wallmapu",
            fontsize=FS_TP, fontweight="bold", fontfamily=FONT,
            color="white", ha="center", va="center",
            transform=_PROJ, zorder=10,
            bbox=dict(boxstyle="round,pad=0.22", fc=WALL_EDGE,
                      alpha=0.88, ec="none"))

    # Santiago
    ax.scatter(-70.65, -33.45, s=28, color="#000000",
               transform=_PROJ, zorder=10, ec="white", lw=0.8)
    ax.text(-70.30, -33.45, "Santiago",
            fontsize=FS_CT, fontfamily=FONT, color=CITY_LBL,
            ha="left", va="center", transform=_PROJ,
            path_effects=stroke())

    # Country labels
    ax.text(-71.0, -26, "Chile", fontsize=FS_CT+1, style="italic",
            fontfamily=FONT, color="#444", ha="center", transform=_PROJ)
    ax.text(-63.5, -38.5, "Argentina", fontsize=FS_CT, style="italic",
            fontfamily=FONT, color="#666", ha="center", transform=_PROJ)

    gl1 = ax.gridlines(draw_labels=True, linewidth=0.4,
                       alpha=0.5, linestyle=":")
    gl1.top_labels = False; gl1.right_labels = False
    gl1.xlabel_style = {"size": FS_AX, "family": FONT}
    gl1.ylabel_style = {"size": FS_AX, "family": FONT}

    ax.set_title("A — Wallmapu and Chilean State and Regional Borders",
                 fontsize=FS_TTL, fontweight="bold",
                 fontfamily=FONT, loc="left", pad=4)

    # ── Legend on Panel A ──────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(fc=ARA_FILL, alpha=0.65, ec=ARA_EDGE,
                       label="La Araucanía region"),
        mpatches.Patch(fc=WALL_FILL, alpha=0.60, ec=WALL_EDGE,
                       label="Wallmapu"),
        Line2D([0],[0], color=REG_EDGE, lw=0.8,
               label="Chilean regional borders"),
        Line2D([0],[0], marker="^", color="w", mfc=VOL_COL,
               mec="white", ms=10, label="Volcano"),
        mpatches.Patch(fc=LAKE_COL, ec=LAKE_EDGE, label="Lake"),
        Line2D([0],[0], marker="o", color="w", mfc="#000000",
               mec="white", ms=9, label="Regional capital (Temuco)"),
        Line2D([0],[0], marker="o", color="w", mfc="#000000",
               mec="white", ms=6, label="Town / village"),
    ]
    leg = ax.legend(handles=legend_handles, loc="lower right",
              fontsize=FS_LK, framealpha=0.93, edgecolor="#ccc",
              title="Legend", title_fontsize=FS_LK+0.5,
              handlelength=1.5, borderpad=0.8,
              prop={"family": FONT, "size": FS_LK})
    leg.set_zorder(30)

    # ══════════════════════════════════════════════════════════════════
    #  Panel B — Araucanía detail
    # ══════════════════════════════════════════════════════════════════
    ax = ax2
    ax.set_extent([ARA["w"], ARA["e"], ARA["s"], ARA["n"]], crs=_PROJ)

    ax.add_feature(cfeature.OCEAN.with_scale("10m"),
                   facecolor=OCEAN, zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("10m"),
                   facecolor=LAND, edgecolor="none", zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"),
                   linewidth=0.6, edgecolor="#777", zorder=6)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"),
                   linewidth=0.9, edgecolor="#999",
                   linestyle="--", zorder=6)

    # Chile country outline — bold
    chile.plot(ax=ax, facecolor="none", edgecolor="#2a2a2a",
               linewidth=1.8, zorder=9, transform=_PROJ)

    # Regional borders
    for _, row in chile_reg.iterrows():
        draw_poly(ax, row.geometry, z=7,
                  fc="none", ec=REG_EDGE, lw=0.6, ls="-")

    # Araucanía fill + border
    for _, row in ara_reg.iterrows():
        draw_poly(ax, row.geometry, z=8,
                  fc=ARA_FILL, ec=ARA_EDGE, lw=1.8, alpha=ARA_ALPHA)

    # Wallmapu subtle tint
    draw_wallmapu_blurred(ax, wallmapu_full, WALL_BBOX, n_steps=8)

    # ── Lakes ──────────────────────────────────────────────────────────
    for _, row in lakes_clip.iterrows():
        g = row.geometry
        if g is None or g.is_empty:
            continue
        draw_poly(ax, g, z=10, fc=LAKE_COL, ec=LAKE_EDGE, lw=0.4, alpha=0.9)

    # ── Volcanoes ──────────────────────────────────────────────────────
    for name, lon, lat, elev in VOLCANOES:
        ax.scatter(lon, lat, marker="^", s=160, color=VOL_COL,
                   edgecolors="white", linewidths=0.8,
                   transform=_PROJ, zorder=12)
        dx, dy, ha = VOL_OFF[name]
        ax.text(lon+dx, lat+dy, f"{name}\n{elev:,} m",
                fontsize=FS_CT, fontfamily=FONT, fontweight="bold",
                color=VOL_LBL, ha=ha, va="center",
                transform=_PROJ, zorder=13,
                path_effects=stroke(3))

    # ── Cities ─────────────────────────────────────────────────────────
    for name, lon, lat, is_cap, dx, dy, ha in CITIES:
        ax.scatter(lon, lat,
                   s=80 if is_cap else 45,
                   color=CITY_CAP,
                   edgecolors="white",
                   linewidths=1.0,
                   transform=_PROJ, zorder=14)
        ax.text(lon+dx, lat+dy, name,
                fontsize=FS_TP if is_cap else FS_CT,
                fontfamily=FONT,
                fontweight="bold" if is_cap else "normal",
                color=CITY_LBL,
                ha=ha, va="center",
                transform=_PROJ, zorder=15,
                path_effects=stroke(3))

    # ── Lake labels ────────────────────────────────────────────────────
    for lon, lat, text in LAKE_LABELS:
        ax.text(lon, lat, text,
                fontsize=FS_LK, fontfamily=FONT,
                color=LAKE_EDGE, style="italic",
                ha="center", va="center",
                transform=_PROJ, zorder=16,
                path_effects=stroke(2))

    # ── Context labels ─────────────────────────────────────────────────
    ax.text(-70.85, -38.9, "Argentina",
            fontsize=FS_CT, fontfamily=FONT, style="italic",
            color="#706050", ha="center",
            transform=_PROJ, path_effects=stroke())
    ax.text(-73.3, -39.6, "Chile",
            fontsize=FS_CT, fontfamily=FONT, style="italic",
            color="#404040", ha="center",
            transform=_PROJ, path_effects=stroke())
    ax.text(-71.15, -37.85, "Andes",
            fontsize=FS_LK, fontfamily=FONT, style="italic",
            color="#6a5030", ha="center", rotation=82,
            transform=_PROJ, path_effects=stroke())

    gl2 = ax2.gridlines(draw_labels=True, linewidth=0.4,
                        alpha=0.5, linestyle=":")
    gl2.top_labels = False; gl2.right_labels = False
    gl2.xlabel_style = {"size": FS_AX, "family": FONT}
    gl2.ylabel_style = {"size": FS_AX, "family": FONT}

    north_arrow(ax2, x=0.96, y=0.05)
    scale_bar(ax2, ARA["w"]+0.18, ARA["s"]+0.22, 50)

    ax2.set_title("B — Glaciated Volcanoes & Communities Studied",
                  fontsize=FS_TTL, fontweight="bold",
                  fontfamily=FONT, loc="left", pad=4)

    # ══════════════════════════════════════════════════════════════════
    #  Leader lines: inset box corners → Panel B edges
    # ══════════════════════════════════════════════════════════════════
    fig.canvas.draw()

    def data_to_fig(ax_obj, lon, lat):
        disp = ax_obj.transData.transform(
            _PROJ.transform_point(lon, lat, _PROJ))
        return fig.transFigure.inverted().transform(disp)

    for (lon_a, lat_a), (lon_b, lat_b) in [
        ((ARA["e"], ARA["n"]), (ARA["w"], ARA["n"])),
        ((ARA["e"], ARA["s"]), (ARA["w"], ARA["s"])),
    ]:
        x0, y0 = data_to_fig(ax1, lon_a, lat_a)
        x1, y1 = data_to_fig(ax2, lon_b, lat_b)
        fig.add_artist(matplotlib.lines.Line2D(
            [x0, x1], [y0, y1],
            transform=fig.transFigure, color="#d32f2f",
            lw=1.0, ls="--", zorder=20, clip_on=False))

    # ── Suptitle & caption ─────────────────────────────────────────────
    fig.suptitle("La Araucanía, Chile — Study Area",
                 fontsize=13, fontweight="bold",
                 fontfamily=FONT, y=0.975)
    fig.text(0.5, 0.01,
             "Volcano coordinates: GVP Smithsonian. "
             "Wallmapu: approx. ancestral Mapuche territory "
             "(Mapocho R. ~33.45°S → S Chiloé ~43.5°S), clipped to land; "
             "faded N/S edges reflect uncertain territorial extent. "
             "Projection: WGS 84.",
             ha="center", fontsize=FS_AX, fontfamily=FONT,
             color="#555", style="italic")

    # ── Save ───────────────────────────────────────────────────────────
    for suffix in [".png", ".pdf"]:
        out = Path(f"Araucania_Study_Area_Map{suffix}")
        kw  = dict(bbox_inches="tight", facecolor="white")
        if suffix == ".png":
            kw["dpi"] = 300
        fig.savefig(out, **kw)
        print(f"✓  {out.resolve()}")
    plt.close(fig)


if __name__ == "__main__":
    import matplotlib
    print("Building Araucanía study area map …\n")
    make_figure()
    print("Done.")